[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Computer Vision - Conditional Diffusion Models

This notebooks applies an image translation model (_Pix2Pix_) using a a Conditional Diffusion generative model.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 08/08/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0131DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing and Computer Vision

# Machine Learning
from sklearn.model_selection import train_test_split

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import torchinfo

from torchmetrics.functional.image import structural_similarity_index_measure
from torchmetrics.functional.regression import r2_score

import torchvision
from torchvision.io import decode_image
from torchvision.transforms import v2 as TorchVisionTrns

# Miscellaneous
import os
import random
import time
from zipfile import ZipFile

# Typing
from typing import Callable, List, Literal, Optional, Tuple
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF = (8, 8)

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataManipulation import DownloadUrl
from DeepLearningBlocks import NNMode

In [ ]:
# General Auxiliary Functions

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """Converts a PyTorch Tensor to a NumPy array."""
    return tZ.squeeze().detach().cpu().numpy()

def TensorImgNumpy( tI: Tensor ) -> NDArray:
    """Converts a CHW image tensor to an HWC NumPy array."""
    return TensorImageNumpy(tI.permute(1, 2, 0))

class SatAerialMapDataset(Dataset):
    def __init__( self, rootFolderPath: str, dataSet: Literal['Train', 'Validation', 'All'], /, *, imgSize: Optional[int] = None, hTrns: Optional[Callable] = None ) -> None:
        """
        Satellite Aerial Map Segmentation Dataset.
        The dataset folder structure:
         - Train
            - 00001.jpg
            - 00002.jpg
            - ...
         - Validation
            - 00001.jpg
            - 00002.jpg
            - ...
        Each image is (600, 1200, 3) where the left (600, 600, 3) is the aerial image and the right (600, 600, 3) is the map image.

        Parameters
        ----------
        rootFolderPath : str
            Path to the folder containing images sets.
        dataSet : Literal['Train', 'Validation', 'All']
            Dataset type to be used. Can be 'Train', 'Validation', or 'All'.
        hTrns : Optional[Callable], optional
            Transform to be applied on the features image (Aerial).
            Should be limited to pixel wise transforms (e.g., normalization, color jitter, etc...).
            By default None.
        """
        super().__init__()

        if dataSet not in ('Train', 'Validation', 'All'):
            raise ValueError("dataSet must be 'Train', 'Validation', or 'All'")

        lDataSets = ['Train', 'Validation'] if dataSet == 'All' else [dataSet]
        lImgFiles = []
        for dataSetName in lDataSets:
            dataSetFolderPath = os.path.join(rootFolderPath, dataSetName)
            if not os.path.isdir(dataSetFolderPath):
                raise FileNotFoundError(f'Dataset folder does not exist: {dataSetFolderPath}')

            lDataSetFiles = sorted(
                os.path.join(dataSetFolderPath, fileName)
                for fileName in os.listdir(dataSetFolderPath)
                if os.path.isfile(os.path.join(dataSetFolderPath, fileName)) and fileName.lower().endswith(('.jpg', '.jpeg', '.png'))
            )
            lImgFiles.extend(lDataSetFiles)

        self._lImgFiles = lImgFiles
        self._imgSize   = imgSize
        self._hTrns     = hTrns

    def __len__( self ) -> int:
        """
        Returns the number of paired images in the dataset.
        """

        return len(self._lImgFiles)

    def __getitem__( self, idx: int ) -> Tuple[Tensor, Tensor]:
        """
        Returns the aerial image and its corresponding map image.

        Parameters
        ----------
        idx : int
            Index of the sample to be fetched.

        Returns
        -------
        Tuple[Tensor, Tensor]
            A tuple containing the aerial image and the map image.
        """
        tPair = decode_image(self._lImgFiles[idx], mode = 'RGB')
        imgWidth = tPair.shape[2]

        imgWidthHalf = imgWidth // 2
        tX = tPair[:, :, :imgWidthHalf]
        tY = tPair[:, :, imgWidthHalf:]

        if self._imgSize is not None:
            tX = TorchVisionTrns.functional.resize(tX, size = (self._imgSize, self._imgSize), interpolation = TorchVisionTrns.InterpolationMode.BILINEAR, antialias = True)
            tY = TorchVisionTrns.functional.resize(tY, size = (self._imgSize, self._imgSize), interpolation = TorchVisionTrns.InterpolationMode.BILINEAR, antialias = True)

        if self._hTrns:
            tX = self._hTrns(tX)

        tY = TorchVisionTrns.functional.to_dtype(tY, torch.float, scale = True)

        return tX, tY

    def SetImageSize( self, imgSize: Optional[int] ) -> None:
        """
        Sets the image size for resizing the aerial and map images.

        Parameters
        ----------
        imgSize : Optional[int]
            The desired image size. If None, no resizing will be applied.
        """
        self._imgSize = imgSize

    def SetTransforms( self, hTrns: Optional[Callable] ) -> None:
        """
        Sets the pixel wise transforms applied to the aerial image.
        """
        self._hTrns = hTrns

* <font color='blue'>(**!**)</font> Inspect `SatAerialMapDataset`: each paired image contains the aerial image on the left and its aligned RGB map on the right.
* <font color='brown'>(**#**)</font> Maps are RGB regression targets, not integer segmentation labels. Noise targets are generated later in the training loop.

## Conditional Diffusion Model

A _Conditional Diffusion Model_ uses extra information at each denoising step.  
This _Condition_ can be a class label, text or an image.

The goal of those 2 models is different:
 - Diffusion Model: Generate a plausible sample.
 - Conditional Diffusion Model: Generate a plausible sample that matches the conditional information.

In this notebook, the condition is an aerial image. The generated sample is its corresponding map.

- **Training:** Add noise to the target map. Predict that noise using the noisy map, the timestep, and the aerial image.
- **Sampling:** Start the map from random noise. Keep the aerial image fixed throughout denoising.
- **Key Distinction:** Only the target map is diffused. The aerial image provides spatial information about roads, buildings, and other features.

The noise prediction objective stays the same. The denoiser gains an extra input: the condition.

Next, _Classifier Free Guidance (CFG)_ lets us adjust the influence of that condition during sampling, without retraining.

### Conditional Diffusion with Classifier Free Guidance

#### Motivation

An unconditional diffusion model can generate a plausible map, but it does not know which roads, buildings, and parks belong to a particular aerial image. A conditional model receives that image and learns to use it. At sampling time, we may want stronger adherence to the source than ordinary conditional sampling provides.

Classifier Free Guidance (CFG) provides a sampling time control over that adherence, without training a separate classifier.  
Here the condition is an image, not a class label; the same idea applies to text and other conditions.

#### Training One Model with Two Tasks

The source aerial image $\boldsymbol{x}$ conditions generation of its aligned target map $\boldsymbol{y}_0$. Only the target is diffused:

$$ \boldsymbol{y}_t = \sqrt{\bar\alpha_t}\boldsymbol{y}_0 + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon}\sim\mathcal N(\boldsymbol{0},\boldsymbol{I}). $$

Here $\alpha_t=1-\beta_t$ and $\bar\alpha_t=\prod_{s=1}^t\alpha_s$. Source and target are scaled from $[0,1]$ to $[-1,1]$ outside the dataset. The aerial image is not diffused.

During training, independently drop the entire aerial condition for each example with probability `conditionDropProb`. Dropping means setting the source channels to zero and the presence mask to zero. Otherwise, the original source is supplied with a mask of one. This distinguishes a missing condition from a real image whose normalized pixels happen to be zero.

The same U-Net learns conditional and unconditional noise prediction with the same noise target and loss:

$$ \mathcal L(\theta)=\mathbb E\left[\left\|\boldsymbol{\epsilon}-\boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\boldsymbol{c})\right\|_2^2\right], \qquad \boldsymbol{c}\in\{\boldsymbol{x},\varnothing\}. $$

```mermaid
flowchart LR
    Clean[Clean target map] --> Forward[Add sampled noise at time t]
    Noise[Sampled noise] --> Forward
    Source[Aerial image] --> Drop{Drop condition?}
    Drop -->|No| Present[Source image and mask one]
    Drop -->|Yes| Null[Zero image and mask zero]
    Forward --> Net[Shared time-conditioned U-Net]
    Present --> Net
    Null --> Net
    Net --> Loss[Noise prediction MSE]
    Noise --> Loss
```

The null-condition task learns what maps generally look like. The conditional task additionally learns how the source changes the denoising prediction. Condition dropout is not dropout of hidden network activations.

#### Intuition at Sampling Time

At the same noisy map and timestep, ask the shared network twice:

- Without the source: what noise should be removed to move toward a plausible map?
- With the source: what noise should be removed to move toward a map compatible with this aerial image?

The difference between these noise predictions captures the effect of supplying the source. CFG amplifies that difference:

$$ \hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\varnothing) + w\left[\boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\boldsymbol{x})-\boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\varnothing)\right]. $$

For $w>1$, this extrapolates beyond the conditional prediction; it is not an average of two generated images. Under the usual noise-to-score conversion, the same combination amplifies the condition-dependent contribution to the score.

```mermaid
flowchart LR
    State[Current noisy map and time] --> Uncond[Shared U-Net with null condition]
    State --> Cond[Shared U-Net with aerial condition]
    Source[Fixed aerial image] --> Cond
    Uncond --> CFG[Combine noise predictions using guidance scale]
    Cond --> CFG
    CFG --> Step[One DDPM reverse step]
    Step --> Next[Less noisy map]
```

- $w=0$: unconditional map generation.
- $w=1$: ordinary conditional DDPM.
- $w>1$: amplify the effect of the aerial condition; excessive guidance can cause artifacts or reduce diversity.

Guidance changes sampling, not the trained weights. Stronger guidance is not guaranteed to improve pixel accuracy, and it does not enforce exact road alignment. The comparison at the end of the notebook keeps both initial and per-step noise fixed so differences reflect the guidance scale.

The guided prediction determines the DDPM reverse mean. The schedule adds fresh Gaussian noise at each reverse step except the last. Every step uses the same network weights; no classifier is trained.

* <font color='brown'>(**#**)</font> This is paired image translation, not the original adversarial Pix2Pix method. It uses no discriminator.
* <font color='brown'>(**#**)</font> Source and target must remain spatially aligned. Do not apply an independent geometric augmentation to only one member of the pair.
* <font color='brown'>(**#**)</font> Reference: [Classifier Free Diffusion Guidance](https://arxiv.org/abs/2207.12598).

In [ ]:
# Parameters

# Data
dataSet    = 'SatAerialToMap'
dataSetUrl = r'https://huggingface.co/datasets/Royi/DataSets/resolve/main/SatAerialToMap.zip'
imgSize = 256
trainNumSamples = None
valNumSamples = 16

# Model
baseCh = 32
useSeparable = False
numDiffSteps = 200
conditionDropProb = 0.1
guidanceScale = 2.0

# Training
batchSize = 8
numWorkers = 0
numEpochs = 200
scoreType = 'R2'

# Validation
valEvery = 10

# Optimizer
ηOpt = 1e-4
tuβ = (0.9, 0.99)
weightDecay = 5e-5
ηSch = 2e-4

# Visualization
numImg = 3
numPlotSteps = 6

## Generate / Load Data

Use the [SatAerialToMap dataset](https://huggingface.co/datasets/Royi/DataSets). Each file contains an aerial/map pair. Reuse existing local files, or download and extract the archive.

We concatenate the supplied `Train` and `Validation` folders and use a seeded split with `valNumSamples = 16`. With `trainNumSamples = None`, all remaining pairs are training samples. Separate dataset instances allow source-only photometric augmentation during training and deterministic validation.

* <font color='brown'>(**#**)</font> This uses a new split, not the archive's original holdout. Nearby geographic tiles may be correlated; geographic splits are preferable for measuring generalization to new locations.

In [ ]:
# Extract Files
# Will create:
# - `FixelCourses/DataSets/SatAerialToMap/Train` - Contains all images.
# - `FixelCourses/DataSets/SatAerialToMap/Validation` - Contains all images.
# Each image is (600, 1200, 3) where the left (600, 600, 3) is the aerial image and the right (600, 600, 3) is the map image.

datasetFolderPath     = os.path.join(DATA_FOLDER_PATH, dataSet)

# Delete existing folders if any
if not os.path.isdir(datasetFolderPath):
    # 1. Download the ZIP file by URL.
    # 2. Extract the ZIP file to the dataset folder path.
    # 3. Delete the ZIP file.

    fileName = os.path.join(DATA_FOLDER_PATH, f'{dataSet}.zip')
    DownloadUrl(dataSetUrl, DATA_FOLDER_PATH)
    with ZipFile(fileName, 'r') as zipFile:
        zipFile.extractall(DATA_FOLDER_PATH) #<! The Zip file contains a folder
    time.sleep(1.0) #<! Wait for the file system to update
    os.remove(fileName)

In [ ]:
# Data Set

oTrnsDisplay = TorchVisionTrns.ToDtype(torch.float32, scale = True)
dsData = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsDisplay)
numSamples = len(dsData)
print(f'Number of paired aerial/map images: {numSamples}')

In [ ]:
# Element of the Data Set

tX, tY = dsData[0]
print(f'Aerial image: {tX.shape}, {tX.dtype}, range [{tX.min():.2f}, {tX.max():.2f}]')
print(f'Target map  : {tY.shape}, {tY.dtype}, range [{tY.min():.2f}, {tY.max():.2f}]')

### Plot the Data

In [ ]:
# Plot Paired Data

hF, mHa = plt.subplots(numImg, 2, figsize = (8, 4 * numImg), squeeze = False)
for sampleIdx in range(numImg):
    tX, tY = dsData[random.randrange(numSamples)]
    for hA, tImage, title in zip(mHa[sampleIdx], (tX, tY), ('Aerial Image', 'Target Map')):
        hA.imshow(TensorImgNumpy(tImage))
        hA.set_title(title)
        hA.axis('off')
hF.tight_layout();

### Augmentation / Transform

Apply photometric augmentations only to the source aerial image. The target map is unchanged and remains in $[0,1]$ at the dataset output.

The training and sampling functions convert image values to $[-1,1]$. Diffusion noise is generated in the training loop, independently of source augmentation.

In [ ]:
# Source Image Transforms

oTrnsTrain = TorchVisionTrns.Compose([
    TorchVisionTrns.ToDtype(torch.float32, scale = True),
    TorchVisionTrns.RandomChoice([
        TorchVisionTrns.RandomGrayscale(p = 1.0),
        TorchVisionTrns.GaussianBlur(7, sigma = (0.1, 1.0)),
        TorchVisionTrns.RandomEqualize(p = 1.0),
        TorchVisionTrns.RandomAutocontrast(p = 1.0),
        TorchVisionTrns.GaussianNoise(sigma = 0.05),
        TorchVisionTrns.RandomErasing(p = 1.0, scale = (0.05, 0.15), ratio = (0.5, 2.0), value = 0, inplace = True),
        TorchVisionTrns.RGB(),
    ], p = [0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.58]),
])
oTrnsVal = TorchVisionTrns.ToDtype(torch.float32, scale = True)

In [ ]:
# Create Training and Validation Datasets

dsTrain = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsTrain)
dsVal   = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsVal)

vIdxTrain, vIdxVal = train_test_split(np.arange(numSamples), test_size = valNumSamples, train_size = trainNumSamples, random_state = seedNum, shuffle = True)

dsTrain = torch.utils.data.Subset(dsTrain, vIdxTrain)
dsVal = torch.utils.data.Subset(dsVal, vIdxVal)

print(f'The training data set contains  : {len(dsTrain):4d} samples.')
print(f'The validation data set contains: {len(dsVal):4d} samples.')

In [ ]:
# Inspect an Augmented Pair

tX, tY = dsTrain[0]
print(f'Source: {tX.shape}, target: {tY.shape}')
hF, vHa = plt.subplots(1, 2, figsize = (8, 4))
for hA, tImage, title in zip(vHa, (tX, tY), ('Augmented Aerial', 'Unchanged Target Map')):
    hA.imshow(TensorImgNumpy(tImage).clip(0, 1))
    hA.set_title(title)
    hA.axis('off')
hF.tight_layout();

In [ ]:
# Forward and Reverse Diffusion Schedule

class DiffusionSchedule:
    def __init__( self, numSteps: int, runDevice: torch.device = torch.device('cpu') ) -> None:
        # Assumes numSteps >= 2. Construct on the same device as the images.
        self.numSteps = numSteps
        vGrid = torch.linspace(0, 1, numSteps + 1, dtype = torch.float64)
        vCurve = torch.cos((vGrid + 0.008) / 1.008 * np.pi / 2).square()
        vBeta = (1 - vCurve[1:] / vCurve[:-1]).clamp(1e-5, 0.999)
        vAlpha = 1 - vBeta
        vAlphaBar = torch.cumprod(vAlpha, dim = 0)
        vAlphaPrev = torch.cat((torch.ones(1, dtype = torch.float64), vAlphaBar[:-1]))
        self.vBeta = vBeta.float().to(runDevice)
        self.vAlphaBar = vAlphaBar.float().to(runDevice)
        self.vVariance = (vBeta * (1 - vAlphaPrev) / (1 - vAlphaBar)).float().to(runDevice)
        self.vCoefClean = (vBeta * vAlphaPrev.sqrt() / (1 - vAlphaBar)).float().to(runDevice)
        self.vCoefNoisy = (vAlpha.sqrt() * (1 - vAlphaPrev) / (1 - vAlphaBar)).float().to(runDevice)

    def AddNoise( self, tClean: Tensor, vTime: Tensor, tNoise: Tensor ) -> Tensor:
        # Assumes BCHW images, matching noise, and integer timestep indices in [0, numSteps).
        tAlphaBar = self.vAlphaBar[vTime].view(-1, 1, 1, 1)
        return tAlphaBar.sqrt() * tClean + (1 - tAlphaBar).sqrt() * tNoise

    def Step( self, tNoisy: Tensor, tNoiseHat: Tensor, stepIdx: int, tNoise: Tensor ) -> Tensor:
        # Assumes matching BCHW tensors and 0 <= stepIdx < numSteps.
        alphaBar = self.vAlphaBar[stepIdx]
        tClean = ((tNoisy - (1 - alphaBar).sqrt() * tNoiseHat) / alphaBar.sqrt()).clamp(-1, 1)
        tMean = self.vCoefClean[stepIdx] * tClean + self.vCoefNoisy[stepIdx] * tNoisy
        return tMean + self.vVariance[stepIdx].sqrt() * tNoise

oDiff = DiffusionSchedule(numDiffSteps)

* <font color='brown'>(**#**)</font> Code index `0` is the first noisy state ($t=1$). The clean target is $\boldsymbol{y}_0$. The cosine schedule makes the final state approximately standard Gaussian.
* <font color='brown'>(**#**)</font> `Step()` clips only the clean-map estimate to $[-1,1]$, then uses it to compute the reverse mean. Reverse variance is $\tilde\beta_t=\beta_t(1-\bar\alpha_{t-1})/(1-\bar\alpha_t)$, which is zero at the final step. Intermediate noisy states are not clipped.
* <font color='red'>(**?**)</font> Why must flips and rotations be applied jointly to the aerial image and map?

### Data Loaders

In [ ]:
# Data Loaders

pinMemory = torch.cuda.is_available()
persWork = numWorkers > 0
prefetchFactor = 2 if persWork else None

dlTrain = torch.utils.data.DataLoader(dsTrain, shuffle = True, batch_size = batchSize, num_workers = numWorkers, pin_memory = pinMemory, drop_last = True, persistent_workers = persWork, prefetch_factor = prefetchFactor)
dlVal = torch.utils.data.DataLoader(dsVal, shuffle = False, batch_size = batchSize, num_workers = numWorkers, pin_memory = pinMemory, drop_last = False, persistent_workers = persWork, prefetch_factor = prefetchFactor)

In [ ]:
# Forward Diffusion of a Target Map

tX, tY = next(iter(dlVal))
print(f'Aerial batch: {tX.shape}, map batch: {tY.shape}')
tClean = (tY[:1] * 2 - 1).repeat(numPlotSteps, 1, 1, 1)
vTime = torch.linspace(0, numDiffSteps - 1, numPlotSteps).long()
tNoisy = oDiff.AddNoise(tClean, vTime, torch.randn_like(tClean))
hF, vHa = plt.subplots(1, numPlotSteps + 2, figsize = (3 * (numPlotSteps + 2), 3))
vHa[0].imshow(TensorImgNumpy(tX[0])); vHa[0].set_title('Aerial Condition')
vHa[1].imshow(TensorImgNumpy(tY[0])); vHa[1].set_title('Clean Map')
for plotIdx, hA in enumerate(vHa[2:]):
    hA.imshow(TensorImgNumpy((tNoisy[plotIdx] + 1) / 2).clip(0, 1))
    hA.set_title(f'Step {vTime[plotIdx].item() + 1}')
for hA in vHa:
    hA.axis('off')
hF.tight_layout();

## Build the Conditional Denoiser

A convolutional U-Net predicts three channels of Gaussian noise. It receives seven input channels: noisy RGB map, normalized RGB aerial image, and a binary condition-presence mask. When the condition is dropped, the aerial channels and mask are zero.

Four downsampling stages capture context; bilinear resizing and skip connections restore spatial detail. GroupNorm supports small batches and mixed noise levels. A sinusoidal timestep embedding is projected into every residual block.

### Bottleneck Attention

**Convolutions are local. Attention is global.**

A $3 \times 3$ convolution mixes each position with its 8 neighbors only.  
Stacking layers and pooling grows the receptive field, yet distant positions interact only through many intermediate layers.

At the bottleneck the $256 \times 256$ image is a $16 \times 16$ grid. Each position summarizes a $16 \times 16$ pixel patch.  
Self attention lets every bottleneck position look at all $256$ positions in a single layer:

1. Normalize the features. Project each position to a query $\boldsymbol{q}$, a key $\boldsymbol{k}$ and a value $\boldsymbol{v}$ using $1 \times 1$ convolutions.
2. Compare each query with all keys: $\operatorname{softmax} \left( \boldsymbol{Q} \boldsymbol{K}^{T} / \sqrt{d} \right)$. Each row is a set of weights over all positions.
3. Replace each position by the weighted average of all values. Add the result to the input (residual).

The block runs once, at the bottleneck, where it is cheap. Its cost grows with the square of the number of positions:
 - $16 \times 16 = 256$ positions: $256^2 \approx 6.5 \cdot 10^{4}$ similarity scores per head.
 - $256 \times 256$ positions: $\left( 6.5 \cdot 10^{4} \right)^2 \approx 4.3 \cdot 10^{9}$. Not feasible at full resolution.

For maps, global mixing helps:
 - A road entering a tile on the left should continue consistently to the right.
 - Water, parks and residential blocks should keep one color across a region.
 - Aerial context from the whole tile informs each local decision.

* <font color='brown'>(**#**)</font> The output projection is zero initialized. The block starts as the identity and learns how much to rely on attention. Early training matches the purely convolutional model.
* <font color='brown'>(**#**)</font> Multi head attention (`numHeads = 4`) splits the channels into groups, each with its own weights over positions. It adds no parameters compared to a single head.
* <font color='brown'>(**#**)</font> The block has no time input. The timestep enters through the residual blocks before and after it.
* <font color='brown'>(**#**)</font> Attention at the $32 \times 32$ level ($1024$ positions) is also common, at $16$ times the compute of the bottleneck block.

In [ ]:
# Time Conditioned U-Net

class TimeBlock(nn.Module):
    def __init__( self, inCh: int, outCh: int, timeDim: int, *, useSeparable: bool = True ) -> None:
        # Assumes outCh is divisible by 8 for GroupNorm.
        super().__init__()
        oConv = nn.Conv2d(inCh, outCh, 3, padding = 1)
        if useSeparable and min(inCh, outCh) >= 64:
            oConv = nn.Sequential(nn.Conv2d(inCh, inCh, 3, padding = 1, groups = inCh, bias = False), nn.Conv2d(inCh, outCh, 1)) #<! Spatial filtering per channel, then channel mixing
        oOut = nn.Conv2d(outCh, outCh, 3, padding = 1)
        if useSeparable and outCh >= 64:
            oOut = nn.Sequential(nn.Conv2d(outCh, outCh, 3, padding = 1, groups = outCh, bias = False), nn.Conv2d(outCh, outCh, 1))
        self.oConv = nn.Sequential(oConv, nn.GroupNorm(8, outCh), nn.SiLU())
        self.oTime = nn.Linear(timeDim, outCh)
        self.oOut = nn.Sequential(nn.GroupNorm(8, outCh), nn.SiLU(), oOut)
        self.oSkip = nn.Conv2d(inCh, outCh, 1) if inCh != outCh else nn.Identity()

    def forward( self, tX: Tensor, mTime: Tensor ) -> Tensor:
        tZ = self.oConv(tX) + self.oTime(mTime)[:, :, None, None]
        return self.oOut(tZ) + self.oSkip(tX)

class AttentionBlock(nn.Module):
    def __init__( self, numCh: int, numHeads: int = 4 ) -> None:
        # Assumes numCh is divisible by numHeads and by 8 for GroupNorm.
        super().__init__()
        self.numHeads = numHeads
        self.oNorm = nn.GroupNorm(8, numCh)
        self.oQKV = nn.Conv2d(numCh, 3 * numCh, 1)
        self.oOut = nn.Conv2d(numCh, numCh, 1)
        nn.init.zeros_(self.oOut.weight) #<! Block starts as the identity
        nn.init.zeros_(self.oOut.bias)

    def forward( self, tX: Tensor ) -> Tensor:
        numB, numCh, numRows, numCols = tX.shape
        tQ, tK, tV = self.oQKV(self.oNorm(tX)).view(numB, 3, self.numHeads, numCh // self.numHeads, numRows * numCols).transpose(-1, -2).unbind(1) #<! Each: B x Heads x (H W) x (C / Heads)
        tZ = F.scaled_dot_product_attention(tQ, tK, tV) #<! Every position attends to all positions
        return tX + self.oOut(tZ.transpose(-1, -2).reshape(numB, numCh, numRows, numCols))

class ConditionalUNet(nn.Module):
    def __init__( self, baseCh: int = 32, timeDim: int = 128, *, useSeparable: bool = False ) -> None:
        # Assumes baseCh is a positive multiple of 8, and timeDim is even and >= 4.
        super().__init__()
        self.vFreq = torch.exp(-np.log(10000.0) * torch.arange(timeDim // 2) / (timeDim // 2 - 1))
        self.oTime = nn.Sequential(nn.Linear(timeDim, timeDim), nn.SiLU(), nn.Linear(timeDim, timeDim))
        self.oEnc1 = TimeBlock(7, baseCh, timeDim, useSeparable = useSeparable)
        self.oEnc2 = TimeBlock(baseCh, 2 * baseCh, timeDim, useSeparable = useSeparable)
        self.oEnc3 = TimeBlock(2 * baseCh, 4 * baseCh, timeDim, useSeparable = useSeparable)
        self.oEnc4 = TimeBlock(4 * baseCh, 8 * baseCh, timeDim, useSeparable = useSeparable)
        self.oMid = TimeBlock(8 * baseCh, 8 * baseCh, timeDim, useSeparable = useSeparable)
        self.oAttn = AttentionBlock(8 * baseCh) #<! Global mixing at the 16 x 16 bottleneck
        self.oUpsample = nn.Upsample(scale_factor = 2, mode = 'bilinear', align_corners = False)
        self.oDec4 = TimeBlock(16 * baseCh, 8 * baseCh, timeDim, useSeparable = useSeparable)
        self.oDec3 = TimeBlock(12 * baseCh, 4 * baseCh, timeDim, useSeparable = useSeparable)
        self.oDec2 = TimeBlock(6 * baseCh, 2 * baseCh, timeDim, useSeparable = useSeparable)
        self.oDec1 = TimeBlock(3 * baseCh, baseCh, timeDim, useSeparable = useSeparable)
        self.oOut = nn.Conv2d(baseCh, 3, 1)

    def forward( self, tNoisy: Tensor, vTime: Tensor, tSource: Tensor, vCondition: Tensor ) -> Tensor:
        # Assumes aligned B x 3 x H x W images, H/W divisible by 16, and B-element time/condition vectors.
        # vCondition is 1 for a present aerial image and 0 for the null condition.
        self.vFreq = self.vFreq.to(tNoisy.device)
        mAngles = vTime.float()[:, None] * self.vFreq[None, :]
        mTime = self.oTime(torch.cat((mAngles.sin(), mAngles.cos()), dim = 1))
        tPresent = vCondition.to(tNoisy.dtype).view(-1, 1, 1, 1)
        tMask = tPresent.expand(-1, 1, *tNoisy.shape[-2:])
        tInput = torch.cat((tNoisy, tSource * tPresent, tMask), dim = 1)
        tEnc1 = self.oEnc1(tInput, mTime)
        tEnc2 = self.oEnc2(F.avg_pool2d(tEnc1, 2), mTime)
        tEnc3 = self.oEnc3(F.avg_pool2d(tEnc2, 2), mTime)
        tEnc4 = self.oEnc4(F.avg_pool2d(tEnc3, 2), mTime)
        tZ = self.oAttn(self.oMid(F.avg_pool2d(tEnc4, 2), mTime))
        for tSkip, oBlock in [(tEnc4, self.oDec4), (tEnc3, self.oDec3), (tEnc2, self.oDec2), (tEnc1, self.oDec1)]:
            tZ = self.oUpsample(tZ)
            tZ = oBlock(torch.cat((tZ, tSkip), dim = 1), mTime)
        return self.oOut(tZ)

In [ ]:
# Model

oModel = ConditionalUNet(baseCh, useSeparable = useSeparable)

# Run device
runDevice = torch.device('cuda:0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'Running on device: {runDevice}')

In [ ]:
# Model Summary

tuInputSize = [tY.shape, (tY.shape[0],), tX.shape, (tX.shape[0],)]
torchinfo.summary(oModel, tuInputSize, col_names = ['kernel_size', 'input_size', 'output_size', 'num_params'], device = runDevice, row_settings = ['depth', 'var_names'])

In [ ]:
# Model Input / Output

oModel = oModel.to(runDevice).eval()
with torch.inference_mode():
    tTest = torch.randn(1, 3, 32, 32, device = runDevice)
    tSource = torch.randn_like(tTest)
    vTestTime = torch.tensor([numDiffSteps // 2], device = runDevice)
    vPresent = torch.ones(1, device = runDevice)
    tNoiseHat = oModel(tTest, vTestTime, tSource, vPresent)
    tNull = oModel(tTest, vTestTime, tSource, torch.zeros_like(vPresent))
    tNullOther = oModel(tTest, vTestTime, tSource + 1, torch.zeros_like(vPresent))
print(f'Predicted RGB noise: {tNoiseHat.shape}')
print(f'Dropped source is ignored: {torch.equal(tNull, tNullOther)}')

## Train the Model

**Learn to predict noise, using the aerial image as a clue.**

For each training batch:

1. Scale aerial images and maps to $[-1,1]$.
2. Pick a random timestep for each map. Add Gaussian noise.
3. Hide some aerial images. This trains the unconditional branch used by CFG.
4. Predict the added noise. Compare it with the known noise using MSE.
5. Backpropagate and update the model.

Each training example uses one sampled timestep, not the full reverse chain. Random timesteps teach the model to handle different noise levels.

The MSE target is the sampled Gaussian noise, not the clean map. We know this target because we generated the noise ourselves.

### Training Budget

**Diffusion denoisers need many iterations, at every noise level.**

Each iteration shows each map at a single random noise level out of `numDiffSteps = 200`.  
One epoch over ~2200 pairs covers each pair once, at one level only.

The `numEpochs = 200` budget gives ~55k iterations at `batchSize = 8`.  
Fewer epochs produce blurry, low contrast maps even when the training loss looks flat.

The `OneCycleLR` scheduler warms up over the first 10% of the epochs to `ηSch = 2e-4`, then anneals to a small final value.  
Peaking early keeps most of the budget at a useful learning rate.

* <font color='brown'>(**#**)</font> The noise loss decreases slowly after the first epochs. Judge progress by the generated map score, not by the loss curve.

### Loss and Task Score

**Training computes noise loss. Validation scores the generated map.**

Every `valEvery = 10` epochs, generate maps for all `valNumSamples = 16` held-out aerial images. Do not compute validation loss or score training images.

For the validation score:

1. Start from Gaussian noise, not a corrupted reference map.
2. Generate the map through all reverse steps, conditioned only on the aerial image.
3. Compare the final map with its reference in $[0,1]$, using `Pix2PixScore` with `scoreType = 'R2'`.

Compute one $R^2$ over the entire 16-map validation set. Keep its order, batch size, sampling seed, and guidance scale fixed. Score at epochs 10, 20, 30, and so on; save local `BestModel.pt` only when a newly computed score improves.

* <font color='brown'>(**#**)</font> The validation set contains only 16 maps, so its score is a limited estimate of generalization. Skipped epochs have no score or checkpoint comparison. A run shorter than `valEvery` produces no checkpoint.
* <font color='brown'>(**#**)</font> A high noise-prediction score would not establish accurate maps. Reference maps enter the task score, never the sampler.

### Automatic Mixed Precision with Autocast

**Use lower precision where it helps. Keep higher precision where it matters.**

Automatic Mixed Precision (AMP) can save GPU memory and utilize the memory throughput more efficiently.  
This enables a faster training.

Operations within `torch.autocast` context are dispatched by precision for each operation:
 * Eligible CUDA operations use `Float16`.  
 * Numerically sensitive operations stay in `Float32`.
 * Model weights remain `Float32`.

For instance, in the forward pass:

 * Stored model weights remain `Float32`.
 * `nn.Conv2d` and `nn.Linear` use `Float16` inputs and temporary `Float16` copies of weights. Their outputs are `Float16`.
 * `nn.GroupNorm` runs in `Float32`.
 * Other operations follow their autocast rules and input types.
 * The final convolution predicts noise in `Float16`. `tNoiseHat.float()` converts it to `Float32` for the loss and reverse updates.
 * Diffusion updates stay in `Float32`.

#### Vanishing Gradients and AMP

`Float16` can round tiny gradients to zero.  
PyTorch's `GradScaler` temporarily enlarges the loss, which also enlarges the gradients.

1. Scale the loss and backpropagate.
2. Unscale the gradients. Then clip their norm.
3. Update the weights. Skip the update if gradients contain infinity or NaN.
4. Adjust the scale for the next iteration.

Scaling protects small gradients.

* <font color='brown'>(**#**)</font> Validation and sampling need no gradient scaling. They do not backpropagate. 
* <font color='brown'>(**#**)</font> AMP changes numerical precision. Yet it does not change the objectives.

In [ ]:
# Loss and Score

def SSIMScore( tYHat: Tensor, tY: Tensor ) -> Tensor:
    """
    Computes the Structural Similarity Index Measure (SSIM) between two images.
    Assumes that the input images are in the range [0, 1].
    """
    return structural_similarity_index_measure(tYHat, tY, data_range = 1.0)

def ImageR2Score( tYHat: Tensor, tY: Tensor ) -> Tensor:
    """
    Computes the R2 score between two images.
    Assumes that the input images are in the range [0, 1].
    """
    return r2_score(tYHat.flatten(), tY.flatten(), multioutput = 'uniform_average')

class Pix2PixScore(nn.Module):
    def __init__( self, scoreType: Literal['SSIM', 'R2'] = 'SSIM' ) -> None:
        """
        Image quality score for image to image regression.

        Parameters
        ----------
        scoreType : Literal['SSIM', 'R2'], optional
            Score function to use, by default 'SSIM'.
        """
        super().__init__()

        match scoreType:
            case 'SSIM':
                self.hScore = SSIMScore
            case 'R2':
                self.hScore = ImageR2Score
            case _:
                raise ValueError('The parameter `scoreType` must be either `SSIM` or `R2`')

    def forward( self, tYHat: Tensor, tY: Tensor ) -> Tensor:
        """
        Computes the selected score between the generated and target images.

        Parameters
        ----------
        tYHat : Tensor
            Generated image tensor (B x C x H x W).
        tY : Tensor
            Target image tensor (B x C x H x W).

        Returns
        -------
        Tensor
            Scalar image quality score.
        """

        return self.hScore(tYHat, tY)

In [ ]:
# Loss and Score

hL = nn.MSELoss()
hS = Pix2PixScore(scoreType = scoreType)
hL = hL.to(runDevice)
hS = hS.to(runDevice)

In [ ]:
# Training / Validation Epoch

def RunDiffusionEpoch( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hL: Callable, oOpt, *, oScaler = None, dropProb: float = 0.1 ) -> float:
    # Assumes a nonempty training loader of aligned RGB pairs in [0, 1].
    epochLoss = 0.0
    numSamples = 0
    numBatches = len(dlData)
    runDevice = next(oModel.parameters()).device
    oModel.train(True)

    for batchIdx, (tX, tY) in enumerate(dlData):
        # With pinned CPU memory, `non_blocking` lets the CPU schedule work while data transfers to CUDA.
        # Less CPU waiting can reduce runtime; operations in the same CUDA stream still wait for the data.
        tX = tX.to(runDevice, non_blocking = True) * 2 - 1 #<! Aerial condition: [0, 1] -> [-1, 1]
        tY = tY.to(runDevice, non_blocking = True) * 2 - 1 #<! Clean target map: [0, 1] -> [-1, 1]
        batchSize = tY.shape[0]
        vTime = torch.randint(oDiff.numSteps, (batchSize,), device = runDevice) #<! One timestep per map
        tNoise = torch.randn(tY.shape, device = runDevice) #<! Known noise target for training loss only
        tNoisy = oDiff.AddNoise(tY, vTime, tNoise)
        vCondition = (torch.rand(batchSize, device = runDevice) >= dropProb).float()

        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tNoiseHat = oModel(tNoisy, vTime, tX, vCondition)
        valLoss = hL(tNoiseHat.float(), tNoise)
        oOpt.zero_grad()
        if oScaler is not None:
            oScaler.scale(valLoss).backward() #<! Protect small gradients from Float16 underflow
            oScaler.unscale_(oOpt) #<! Unscale before clipping
            nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
            oScaler.step(oOpt)
            oScaler.update()
        else:
            valLoss.backward()
            nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
            oOpt.step()

        epochLoss += batchSize * valLoss.detach().item()
        numSamples += batchSize
        print(f'\rTrain Loss - Iteration: {(batchIdx + 1):3d} / {numBatches}, Loss: {valLoss:.6f}', end = '')

    print('', end = '\r')
    return epochLoss / numSamples

@torch.inference_mode()
def EvaluateDiffusionModel( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hS: Callable, *, guidanceScale: float = 2.0, sampleSeed: int = 512 ) -> float:
    # Assumes a nonempty, fixed-order loader with images in [0, 1].
    runDevice = next(oModel.parameters()).device
    lGenerated, lTarget = [], []
    for batchIdx, (tX, tY) in enumerate(dlData):
        tGenerated, _, _ = SampleMaps(oModel, oDiff, tX, runDevice, guidanceScale = guidanceScale, sampleSeed = sampleSeed + batchIdx, numFrames = 0) #<! Generate from noise and aerial images only
        lGenerated.append(tGenerated)
        lTarget.append(tY.cpu()) #<! Reference maps are used only by the score
        print(f'\rVal Map Score - Batch: {batchIdx + 1:3d} / {len(dlData)}', end = '')
    print('', end = '\r')
    return hS(torch.cat(lGenerated), torch.cat(lTarget)).item() #<! One score over all supplied validation maps

def TrainDiffusionModel( oModel: nn.Module, oDiff: DiffusionSchedule, dlTrain, dlVal, oOpt, numEpoch: int, hL: Callable, hS: Callable, *, oSch = None, oScaler = None, dropProb: float = 0.1, guidanceScale: float = 2.0, valEvery: int = 5, sampleSeed: int = 512 ) -> Tuple[nn.Module, List[float], List[int], List[float], List[float]]:
    # Assumes nonempty loaders and numEpoch > 0. Only scheduled epochs generate validation maps.
    if valEvery < 1:
        raise ValueError('valEvery must be positive')
    lTrainLoss, lLearnRate = [], []
    lValEpoch, lValScore = [], []
    bestScore = -float('inf')

    for epochIdx in range(numEpoch):
        startTime = time.time()
        learnRate = oOpt.param_groups[0]['lr']
        trainLoss = RunDiffusionEpoch(oModel, oDiff, dlTrain, hL, oOpt, oScaler = oScaler, dropProb = dropProb)
        scoreEpoch = (epochIdx + 1) % valEvery == 0
        if scoreEpoch:
            valScr = EvaluateDiffusionModel(oModel, oDiff, dlVal, hS, guidanceScale = guidanceScale, sampleSeed = sampleSeed)
            lValEpoch.append(epochIdx + 1)
            lValScore.append(valScr)
        if oSch is not None:
            oSch.step()
        epochTime = time.time() - startTime

        lTrainLoss.append(trainLoss)
        lLearnRate.append(learnRate)
        print(f'Epoch {(epochIdx + 1):4d} / {numEpoch}', end = '')
        print(f' | Train Noise Loss: {trainLoss:6.3f}', end = '')
        if scoreEpoch:
            print(f' | Val Score: {valScr:6.3f}', end = '')
        print(f' | Epoch Time: {epochTime:5.2f}', end = '')

        if scoreEpoch and valScr > bestScore: #<! Never checkpoint on an unscored epoch
            bestScore = valScr
            try:
                dCheckPoint = {'Model': oModel.state_dict(), 'Optimizer': oOpt.state_dict()}
                if oSch is not None:
                    dCheckPoint['Scheduler'] = oSch.state_dict()
                torch.save(dCheckPoint, 'BestModel.pt')
                print(' | <-- Checkpoint!', end = '')
            except OSError as oError:
                print(f' | <-- Failed: {oError}', end = '')
        print(' |')

    return oModel, lTrainLoss, lValEpoch, lValScore, lLearnRate

### Generate Maps with CFG

Initialize a target-shaped Gaussian tensor and keep the aerial image fixed. At every step, predict conditional and unconditional noise, combine them using `guidanceScale`, and apply the DDPM posterior update.

`SampleMaps()` takes only aerial images, not target maps. It returns generated RGB maps in $[0,1]$ and display snapshots for the first image. Ground-truth maps are used only for evaluation.

* <font color='red'>(**?**)</font> Why would CFG fail if the model were never trained with dropped conditions?
* <font color='brown'>(**#**)</font> This is a full DDPM chain. Do not skip timesteps to accelerate sampling without changing to an appropriate sampler such as DDIM.

In [ ]:
# Classifier-Free Guided DDPM Sampling

def PredictGuidedNoise( oModel: nn.Module, tNoisy: Tensor, vTime: Tensor, tSource: Tensor, guidanceScale: float ) -> Tensor:
    vPresent = torch.ones(len(tNoisy), device = tNoisy.device) #<! One condition-presence flag per image
    if guidanceScale == 1: #<! Ordinary conditional prediction needs only one model call
        return oModel(tNoisy, vTime, tSource, vPresent).float()
    tNoiseNull = oModel(tNoisy, vTime, tSource, torch.zeros_like(vPresent)).float() #<! Zero mask hides the aerial image
    if guidanceScale == 0:
        return tNoiseNull
    tNoiseCond = oModel(tNoisy, vTime, tSource, vPresent).float()
    return tNoiseNull + guidanceScale * (tNoiseCond - tNoiseNull) #<! Amplify the condition's effect on predicted noise

@torch.inference_mode()
def SampleMaps( oModel: nn.Module, oDiff: DiffusionSchedule, tSource: Tensor, runDevice: torch.device, *, guidanceScale: float = 2.0, sampleSeed: int = 512, numFrames: int = 6 ) -> Tuple[Tensor, list, list]:
    """Generate maps from BCHW aerial images in [0, 1]; numFrames = 0 disables snapshots."""
    oModel.eval()
    tSource = tSource.to(runDevice) * 2 - 1 #<! Same condition normalization as training
    oGen = torch.Generator(device = runDevice).manual_seed(sampleSeed)
    tNoisy = torch.randn(tSource.shape, device = runDevice, generator = oGen) #<! No target image is supplied
    lFrames, lSteps = [], []
    if numFrames > 0:
        lFrames.append(((tNoisy[:1].cpu() + 1) / 2).clamp(0, 1))
        lSteps.append(oDiff.numSteps)
    vSaveSteps = np.linspace(oDiff.numSteps, 0, numFrames, dtype = int)
    for stepIdx in reversed(range(oDiff.numSteps)):
        vTime = torch.full((len(tSource),), stepIdx, device = runDevice, dtype = torch.long)
        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tNoiseHat = PredictGuidedNoise(oModel, tNoisy, vTime, tSource, guidanceScale) #<! Same aerial image at every step
        tNoise = torch.randn(tNoisy.shape, device = runDevice, generator = oGen) if stepIdx > 0 else torch.zeros_like(tNoisy)
        tNoisy = oDiff.Step(tNoisy, tNoiseHat, stepIdx, tNoise) #<! Reverse update stays in Float32
        if stepIdx in vSaveSteps:
            lFrames.append(((tNoisy[:1].cpu() + 1) / 2).clamp(0, 1))
            lSteps.append(stepIdx)
    return ((tNoisy.cpu() + 1) / 2).clamp(0, 1), lFrames, lSteps

In [ ]:
# Optimizer Related

oDiff = DiffusionSchedule(numDiffSteps, runDevice)
oOpt = torch.optim.AdamW(oModel.parameters(), lr = ηOpt, betas = tuβ, weight_decay = weightDecay)
oSch = torch.optim.lr_scheduler.OneCycleLR(oOpt, max_lr = ηSch, total_steps = numEpochs, pct_start = 0.1, div_factor = 10, final_div_factor = 20) #<! Warm up from 2e-5, peak at epoch 20, end at 1e-6
oScaler = torch.amp.GradScaler('cuda', enabled = runDevice.type == 'cuda')

In [ ]:
# Training Loop

oModel = oModel.to(runDevice)
_, lTrainLoss, lValEpoch, lValScore, lLearnRate = TrainDiffusionModel(oModel, oDiff, dlTrain, dlVal, oOpt, numEpochs, hL, hS, oSch = oSch, oScaler = oScaler, dropProb = conditionDropProb, guidanceScale = guidanceScale, valEvery = valEvery, sampleSeed = seedNum)

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (18, 5))
vHa = np.ravel(vHa)
vEpoch = np.arange(1, len(lTrainLoss) + 1)

hA = vHa[0]
hA.plot(vEpoch, lTrainLoss, lw = 2, label = 'Train')
hA.set_title('Training Noise Prediction Loss')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lValEpoch, lValScore, 'o-', lw = 2, label = 'Validation')
hA.set_title(f'Generated Map Score ({len(dlVal.dataset)} Maps)')
hA.set_xlabel('Epoch')
hA.set_ylabel(scoreType)
hA.legend()

hA = vHa[2]
hA.plot(vEpoch, lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

In [ ]:
# Load the Model

dModel = torch.load('BestModel.pt', map_location = runDevice, weights_only = True)
oModel.load_state_dict(dModel['Model'])
print('Model loaded from: BestModel.pt')

oDiff = DiffusionSchedule(numDiffSteps, runDevice)
oModel.eval();

In [ ]:
# Compute the Score on the Validation Set

valScore = EvaluateDiffusionModel(oModel, oDiff, dlVal, hS, guidanceScale = 2, sampleSeed = seedNum)
print(f'Validation Score ({len(dlVal.dataset)} Maps): {valScore:.4f}')

In [ ]:
# Aerial Image -> Conditional DDPM -> Map

lPairs = [dsVal[sampleIdx] for sampleIdx in range(min(numImg, len(dsVal)))]
tSource = torch.stack([tuPair[0] for tuPair in lPairs])
tTarget = torch.stack([tuPair[1] for tuPair in lPairs])
tGenerated, lFrames, lSteps = SampleMaps(oModel, oDiff, tSource, runDevice, guidanceScale = guidanceScale, sampleSeed = seedNum, numFrames = numPlotSteps)

hF, mHa = plt.subplots(len(lPairs), 3, figsize = (12, 4 * len(lPairs)), squeeze = False)
for sampleIdx, vHa in enumerate(mHa):
    for hA, tImage, title in zip(vHa, (tSource[sampleIdx], tTarget[sampleIdx], tGenerated[sampleIdx]),
                               ('Aerial Image', 'Target Map', f'Generated Map (CFG {guidanceScale:g})')):
        hA.imshow(TensorImgNumpy(tImage))
        hA.set_title(title)
        hA.axis('off')
hF.tight_layout()

mae = F.l1_loss(tGenerated, tTarget).item()
ssim = structural_similarity_index_measure(tGenerated, tTarget, data_range = 1.0).item()
r2 = hS(tGenerated, tTarget).item()
print(f'Preview only ({len(lPairs)} validation pairs): MAE = {mae:.4f}, SSIM = {ssim:.4f}, R2 = {r2:.4f}')

In [ ]:
# Follow the Reverse Process

tCondition, tReference = dsVal[0]
tMap, lFrames, lSteps = SampleMaps(oModel, oDiff, tCondition[None], runDevice, guidanceScale = guidanceScale, sampleSeed = seedNum, numFrames = numPlotSteps) #<! Rebuild frames for this exact aerial image
numPanels = len(lFrames) + 2
numCols = 4
numRows = (numPanels + numCols - 1) // numCols
hF, mHa = plt.subplots(numRows, numCols, figsize = (4 * numCols, 4 * numRows), squeeze = False)
vHa = mHa.ravel()
vHa[0].imshow(TensorImgNumpy(tCondition))
vHa[0].set_title('Aerial Condition (Fixed)')
vHa[1].imshow(TensorImgNumpy(tReference))
vHa[1].set_title('Reference Map (Not an Input)')
for hA, tFrame, stepIdx in zip(vHa[2:], lFrames, lSteps):
    hA.imshow(TensorImgNumpy(tFrame[0]))
    hA.set_title('Generated Map' if stepIdx == 0 else f'Reverse Steps Remaining: {stepIdx}')
for hA in vHa:
    hA.axis('off')
hF.tight_layout()

# Compare Guidance with Identical Initial and Per-Step Noise
lGuidance = [0.0, 1.0, 2.0]
hF, vHa = plt.subplots(1, len(lGuidance) + 2, figsize = (4 * (len(lGuidance) + 2), 4))
vHa[0].imshow(TensorImgNumpy(tCondition)); vHa[0].set_title('Aerial Condition'); vHa[0].axis('off')
vHa[1].imshow(TensorImgNumpy(tReference)); vHa[1].set_title('Reference Map'); vHa[1].axis('off')
for hA, scale in zip(vHa[2:], lGuidance):
    tMap, _, _ = SampleMaps(oModel, oDiff, tCondition[None], runDevice, guidanceScale = scale, sampleSeed = seedNum, numFrames = 0)
    hA.imshow(TensorImgNumpy(tMap[0]))
    hA.set_title(f'CFG Scale = {scale:g}')
    hA.axis('off')
hF.tight_layout();

* <font color='blue'>(**!**)</font> Change `guidanceScale` without retraining. Compare 0, 1, 2, and 4 with the same seed and aerial image.
* <font color='red'>(**?**)</font> Does stronger guidance always improve road alignment? Look for color saturation, artifacts, and missing small structures.
* <font color='green'>(**@**)</font> Compare multiple seeds for one aerial image to inspect diversity. Not every generated difference represents calibrated geographic uncertainty.
* <font color='green'>(**@**)</font> Compare generated-map R2 across guidance scales using the same validation pairs and sampling seed. The preview metrics cover only displayed pairs; periodic validation scores cover the entire 16-map validation set.
* <font color='green'>(**@**)</font> Swap or shuffle the aerial conditions with identical sampling noise to check whether the generated maps follow the source. Compare CFG 1 with stronger guidance before attributing artifacts to the architecture.
* <font color='brown'>(**#**)</font> Diffusion is a generative model, not a guarantee of accurate cartography. Do not treat generated map details as verified geographic facts.